<a href="https://colab.research.google.com/github/A2-ashish/A2-ashish/blob/main/Kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

# ==============================
# 1. Load Data
# ==============================

train_df = pd.read_csv("/kaggle/input/.../train.csv")
test_df = pd.read_csv("/kaggle/input/.../test.csv")

TARGET = "target"
ID_COL = "Id"

X = train_df.drop([TARGET], axis=1)
y = train_df[TARGET]

# ==============================
# 2. Detect Feature Types
# ==============================

numeric_features = X.select_dtypes(include=np.number).columns
categorical_features = X.select_dtypes(include="object").columns

# ==============================
# 3. Pipelines
# ==============================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# ==============================
# 4. Models
# ==============================

lgb_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", LGBMRegressor(n_estimators=1500, learning_rate=0.02))
])

xgb_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", XGBRegressor(n_estimators=1500, learning_rate=0.02))
])

# ==============================
# 5. Train / Validation
# ==============================

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lgb_pipeline.fit(X_train, y_train)
xgb_pipeline.fit(X_train, y_train)

pred1 = lgb_pipeline.predict(X_val)
pred2 = xgb_pipeline.predict(X_val)

val_pred = (pred1 + pred2) / 2

rmse = np.sqrt(mean_squared_error(y_val, val_pred))
print("Validation RMSE:", rmse)

# ==============================
# 6. Train Final Models
# ==============================

lgb_pipeline.fit(X, y)
xgb_pipeline.fit(X, y)

# ==============================
# 7. Test Prediction
# ==============================

test_pred = (
    lgb_pipeline.predict(test_df) +
    xgb_pipeline.predict(test_df)
) / 2

# ==============================
# 8. Kaggle Submission
# ==============================

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: test_pred
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created!")